In [ ]:
from scMM.file.io import load_single_file, sum_spec, extract_peaks, align_frame, save_spectra
from scMM.plot.msplot import plot_spectrum
exp, metadata = load_single_file("/home/zby/scMM/data/3d-models/20260323-yz-30mM/0907.mzML")
print("load_finished")
spec = sum_spec(exp, resolution_200=35000)

load_finished


In [3]:
import matplotlib.pyplot as plt
fig, ax = plot_spectrum(exp[164], mz_range=(100, 1000), figsize=(6, 6), linewidth=1, color='#B44257')
plt.xticks(size=13)
plt.yticks(size=13)
plt.savefig("/home/zby/src/.tmp/spectrum.svg")

In [4]:
import numpy as np
peaks = extract_peaks(spec)[0]
print(peaks)
print(peaks[np.argmin(np.abs(peaks - 734.6))])

[100.1110241  100.1130441  100.83609924 ... 997.53172564 998.53870359
 999.54846855]
734.5924119314657


In [3]:
from scMM.util.peak import filter_spectrum
spec = filter_spectrum(spec, snr_threshold=10.0)
peaks = extract_peaks(spec)[0]
print(peaks)
print(peaks[np.argmin(np.abs(peaks - 734.6))])

[100.1110241  100.1130441  100.83609924 ... 997.53172564 998.53870359
 999.54846855]
734.5924119314657


In [6]:
align_frame(exp, peaks)

(       100.111024  100.113044  100.836099  100.947734  101.723461  \
 frame                                                               
 0             0.0         0.0         0.0         0.0         0.0   
 1             0.0         0.0         0.0         0.0         0.0   
 2             0.0         0.0         0.0         0.0         0.0   
 3             0.0         0.0         0.0         0.0         0.0   
 4             0.0         0.0         0.0         0.0         0.0   
 ...           ...         ...         ...         ...         ...   
 28651         0.0         0.0         0.0         0.0         0.0   
 28652         0.0         0.0         0.0         0.0         0.0   
 28653         0.0         0.0         0.0         0.0         0.0   
 28654         0.0         0.0         0.0         0.0         0.0   
 28655         0.0         0.0         0.0         0.0         0.0   
 
          101.724912  101.834718   101.836451  101.947904  102.128263  ...  \
 frame    

In [1]:
from scMM.file.data import CyESIData
import logging
logging.basicConfig(level=logging.INFO)

data = CyESIData.load_from_filelist("/home/zby/scMM/data/3d-models/test", ref_mz = 734.5929, prominence_ratio = 0.01)
data.save("/home/zby/scMM/data/3d-models/test_result")

INFO:root:Detected files in targeted directory: ['/home/zby/scMM/data/3d-models/test/0907.mzML', '/home/zby/scMM/data/3d-models/test/0900.mzML']
INFO:root:Summing MS Spectrometry, resolution=35000, resample points per FWHM=5.0...
INFO:root:Performing summed-MS denoising, snr_threshold=10.0...
INFO:root:Performing summed-MS peak picking...
INFO:root:Building data container class...
INFO:root:Performing denoising and cell peak picking...


FileExistsError: [Errno 17] File exists: '/home/zby/scMM/data/3d-models/test_result/test'

In [ ]:
from scMM.plot.msplot import save_hook
from scMM.file.data import CyESIData
data = CyESIData.load_from_file("/home/zby/scMM/data/3d-models/20260329-yz-0mM/0905.mzML", ref_mz = 734.5929,
             debug_hook = save_hook)
#data.save("/home/zby/scMM/data/3d-models/test_result")

In [5]:
import pickle
import numpy as np
with open(".tmp/find_cells.pkl", "rb") as fp:
    file = pickle.load(fp)["data"]

cell_signal, baseline, cell_idx = file["signal"], file["baseline"], file["cell_idx"]
mz_axis = cell_signal.columns.values.astype(float)
import numpy as np
import matplotlib.pyplot as plt
print(np.argmin(np.abs(mz_axis - 734.5929)), mz_axis[np.argmin(np.abs(mz_axis - 734.5929))])
print(np.argmin(np.abs(mz_axis - 423.1)), mz_axis[np.argmin(np.abs(mz_axis - 423.1))])

2721 734.5922047873896
1796 423.1988535099586


In [6]:
index = 2721
frames = np.arange(0, 10000)
cell_idx = np.isin(frames, cell_idx)
plt.figure(figsize=(14, 6))
y = cell_signal.iloc[frames, index]

from matplotlib.ticker import AutoMinorLocator
ax = plt.gca()
ax.set_xlim(0, np.max(frames))
ax.set_ylim(0, np.max(y) * 1.05)
ax.xaxis.set_minor_locator(AutoMinorLocator(10))
ax.set_xticks(np.arange(0, np.max(frames) + 1, 500))
ax.tick_params(axis='x', which='minor', length=3)
ax.tick_params(axis='x', which='major', length=5)

plt.plot(frames, y, color = "#E28962", linewidth =0.5, label="EIC at m/z = 734.5929")
plt.scatter(frames[cell_idx], y[cell_idx], color = "black", s = 3, label="cells")
plt.xticks(size=13)
plt.yticks(size=13)
plt.legend(fontsize=20)
plt.xlabel("Frame", size=22)
plt.ylabel("Intensity", size=22)
plt.savefig("/home/zby/src/.tmp/signal.svg")

In [7]:
frames = np.arange(0, 10000)
cell_idx = np.isin(frames, cell_idx)
tic = cell_signal.iloc[frames].sum(axis=1)

plt.figure(figsize=(14, 6))
ax = plt.gca()
ax.set_xlim(0, np.max(frames))
ax.set_ylim(0, np.max(tic) * 1.05)
plt.plot(frames, tic, color = "#5A9DA4", linewidth=0.5, label="TIC")
plt.xticks([])
plt.yticks(size=13)
plt.legend(fontsize=20)
plt.ylabel("Intensity", size=22)
plt.savefig("/home/zby/src/.tmp/tic.svg")

In [2]:
from scMM.file.batch import batch_process, concat
import logging
logging.basicConfig(level=logging.INFO)
batch_process("/home/zby/scMM/data/3d-models/test", 
              "/home/zby/scMM/data/3d-models/test_result", 
              ref_mz = 734.5929)
concat("/home/zby/scMM/data/3d-models/test_result",
       "/home/zby/scMM/data/3d-models/test_concat_result",
       ref_idx = 0,
       ppm_tol = 5.0,
       mz_merge_options = "union")

INFO:root:Saving processed data to /home/zby/scMM/data/3d-models/test_result/0907...
INFO:root:Saving processed data to /home/zby/scMM/data/3d-models/test_result/0900...
INFO:root:Failed to load directory /home/zby/scMM/data/3d-models/test_result/0900: type object 'CyESIData' has no attribute 'load_from_processed'
INFO:root:Failed to load directory /home/zby/scMM/data/3d-models/test_result/0907: type object 'CyESIData' has no attribute 'load_from_processed'


IndexError: list index out of range